In [ ]:
#@title Install OmniVoice + Dependencies
%cd /content/

# Clean old files
!rm -rf omnivoice-colab
!rm -rf OmniVoice

# Clone your repo
!git clone https://github.com/hahyt6644-gif/omnivoice-colab.git

%cd /content/omnivoice-colab

# Clone MAIN OmniVoice repo (official)
!git clone https://github.com/k2-fsa/OmniVoice.git

# Install ffmpeg
!apt-get update -qq
!apt-get install -y ffmpeg openssh-client

# Install Python dependencies
!pip install -q -r colab.txt

# Extra packages
!pip install -q \
fastapi \
uvicorn[standard] \
python-multipart \
httpx \
requests \
aiofiles \
nest-asyncio \
python-dotenv \
pydub \
soundfile \
librosa

# GPU Check
import torch

print("========== SYSTEM INFO ==========")

if torch.cuda.is_available():
    print("✅ GPU ENABLED")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "❌ GPU NOT ENABLED!\n"
        "Runtime → Change runtime type → GPU"
    )

# Fix imports
import sys
sys.path.append('/content/omnivoice-colab/OmniVoice')

from IPython.display import clear_output
clear_output()

print("✅ Installation complete!")

In [ ]:
#@title Run OmniVoice + Pinggy Tunnel
%cd /content/omnivoice-colab

import subprocess
import requests
import threading
import time
import re
import os

WEBHOOK_URL = "https://itz-saver.itz-dev.workers.dev/update"

# Kill previous processes
!pkill -f "python app.py" || true
!pkill -f "ssh -p 443" || true

print("🚀 Starting OmniVoice...")

# Start OmniVoice server
server = subprocess.Popen(
    ["python", "app.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Wait for server
time.sleep(20)

def start_pinggy():

    while True:

        print("\n🌍 Starting Pinggy Tunnel...\n")

        process = subprocess.Popen(
            [
                "ssh",
                "-p", "443",
                "-R0:localhost:7860",
                "a.pinggy.io",
                "-o", "StrictHostKeyChecking=no",
                "-o", "ServerAliveInterval=30",
                "-o", "ServerAliveCountMax=3"
            ],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True
        )

        current_url = None

        while True:
            line = process.stdout.readline()

            if not line:
                break

            print(line.strip())

            match = re.search(
                r"https://[a-zA-Z0-9\-]+\.pinggy\.link",
                line
            )

            if match:
                url = match.group(0)

                if url != current_url:
                    current_url = url

                    clone_api = f"{url}/api/clone"
                    design_api = f"{url}/api/design"
                    ui_url = f"{url}/ui"

                    print("=" * 60)
                    print("✅ NEW PUBLIC URL")
                    print(url)
                    print("\n🎙 Clone API:")
                    print(clone_api)
                    print("\n🎨 Design API:")
                    print(design_api)
                    print("\n🖥 UI:")
                    print(ui_url)
                    print("=" * 60)

                    payload = {
                        "url": url,
                        "clone_api": clone_api,
                        "design_api": design_api,
                        "ui": ui_url
                    }

                    try:
                        response = requests.post(
                            WEBHOOK_URL,
                            json=payload,
                            timeout=10
                        )

                        print(
                            f"✅ Webhook Sent: "
                            f"{response.status_code}"
                        )

                    except Exception as e:
                        print("❌ Webhook Error:", e)

        print(
            "\n⚠ Tunnel disconnected."
            " Reconnecting in 5 seconds..."
        )

        time.sleep(5)

threading.Thread(
    target=start_pinggy,
    daemon=True
).start()

print("✅ OmniVoice Running!")

### Usage

1. **Enable GPU:** `Runtime` → `Change runtime type` → `GPU`
2. Run **Cell 1**
3. Run **Cell 2**

**You’ll get:**
- `https://xxxxx.pinggy.link/ui`
- `https://xxxxx.pinggy.link/api/clone`
- `https://xxxxx.pinggy.link/api/design`

And every new URL auto-sends to:
`https://itz-saver.workers.dev/update`

*Auto reconnect works if Pinggy disconnects/60 min expires.*